# Exercise 3.1 — One workflow, two models (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day3/day3-01-one-workflow.html>.*

*Open it in Jupyter, VS Code or Positron and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

In R, you can fit a linear regression directly with `lm()`. In this practical
you will use `lm()` as the engine inside the familiar tidymodels steps:
**specify** a model, **fit** it on training patients, and call **`predict()`**
for held-out patients. In Python the same three steps are
`LinearRegression()`, `.fit()` and `.predict()` from scikit-learn.

- **Part 1.** Predict blood glucose from age. Calculate the prediction for a
  45-year-old by hand from the fitted equation, check it with `predict()`, and
  see the point on the fitted line. Compare predictions with the measured
  glucose of the test patients, then refit with different training patients
  to see how the line and the predictions change.
- **Part 2.** Add BMI, repeat the hand calculation, and find the prediction on
  the fitted plane in a 3D plot you can turn.
- **Part 3.** Keep the same steps, training patients and predictors, and fit a
  decision tree instead. Follow the same patient through the splits, read the
  prediction from the leaf, check it with `predict()`, and compare the tree's
  stepped surface with the regression plane.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text

`data/glucose_bmi_study.csv` holds 77 patients with three columns: age in
years, BMI in kg/m² and blood glucose in mg/dL. If you do not have it yet, [**download `glucose_bmi_study.csv`**](data/glucose_bmi_study.csv)
into your `data` folder.

In [ ]:
glu = pd.read_csv("data/glucose_bmi_study.csv")

glu.describe()

Split the 77 patients into training and test patients, as on Day 1.

In [ ]:
train, test = train_test_split(glu, test_size=0.2, random_state=42)

print(len(train), "training patients,", len(test), "test patients")

> **The numbers differ from the R tab**
>
> Python and R shuffle with different random number generators, so
> `random_state=42` and `set.seed(42)` do not choose the same training patients.
> Expect different numbers from the R tab; the steps are the same.

### Part 1 — Predict glucose from age

The same three steps as on Day 1: specify the model, fit it on the training
patients, predict.

In [ ]:
# 1. Specify: linear regression
line_model = LinearRegression()

# 2. Fit on the training patients: predictors (a table) and outcome
line_model.fit(train[["age"]], train["glucose"])

# the learned parameters
print("intercept:", line_model.intercept_)
print("slope:", line_model.coef_)

The fitted equation is

$$\widehat{\text{glucose}} = \hat\beta_0 + \hat\beta_1 \times \text{age}$$

#### Task 1a — A 45-year-old, by hand

Read the two parameters printed above, round them to two decimals, and
calculate the predicted glucose for a 45-year-old.

> **Stuck?**
>
> The intercept is $\hat\beta_0$. The slope, $\hat\beta_1$, multiplies the age.

> **Worked answer**

In [ ]:
predicted_45 = 100.82 + 0.49 * 45
predicted_45

Now check it with `predict()`, which uses the unrounded parameters:

In [ ]:
# predict() wants a table: one row, one column per predictor
line_model.predict(pd.DataFrame({"age": [45]}))

The two agree to within about a tenth: the difference is only the rounding.

In [ ]:
ages_line = np.array([20, 82])
fig, ax = plt.subplots()
ax.scatter(train["age"], train["glucose"], color="steelblue")
ax.plot(ages_line, line_model.intercept_ + line_model.coef_[0] * ages_line,
        "r--", linewidth=1)
ax.scatter([45], line_model.predict(pd.DataFrame({"age": [45]})),
           color="red", s=60, zorder=3)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Glucose (mg/dL)")
ax.set_title("Training patients, the fitted line, and the 45-year-old")
plt.show()

#### Task 1b — Predictions against measured glucose

The test patients were not used to fit the line. Predict their glucose and put
it next to what was measured.

In [ ]:
test_results = test.copy()
test_results["predicted"] = line_model.predict(test[["age"]])
test_results["gap"] = test_results["glucose"] - test_results["predicted"]

test_results

In [ ]:
fig, ax = plt.subplots()
ax.plot(ages_line, line_model.intercept_ + line_model.coef_[0] * ages_line,
        "r--", linewidth=1)
ax.vlines(test_results["age"], test_results["glucose"],
          test_results["predicted"], color="grey")
ax.scatter(test_results["age"], test_results["glucose"], color="darkorange")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Glucose (mg/dL)")
ax.set_title("Test patients: measured (orange) and predicted (on the line)")
plt.show()

> **Think about it**
>
> 1. How far are the predictions from the measured glucose, typically?
> 2. All 16 predictions lie in a narrow band. Why?

> **Worked answer**

In [ ]:
print(test_results["gap"].abs().mean())
print(test_results["predicted"].min(), test_results["predicted"].max())

> 1. On average about 22 mg/dL away, in either direction; some patients are
>    much further.
> 2. The slope is small, about half a mg/dL per year, so even a 60-year
>    difference in age moves the prediction by about 30 mg/dL, while measured
>    glucose varies far more. Age explains very little of the differences
>    between these patients, so the line stays close to the average glucose.
>
> With 16 test patients, treat any summary of the gaps as rough.

#### Task 1c — Different training patients

Split again with a different seed, refit, and compare.

> **Stuck?**
>
> Nothing about the model changes. Only the training patients change, and the
> patient you predict for is the same 45-year-old.

> **Worked answer**

In [ ]:
train_1, test_1 = train_test_split(glu, test_size=0.2, random_state=1)

line_model_1 = LinearRegression()
line_model_1.fit(train_1[["age"]], train_1["glucose"])

print(line_model_1.intercept_, line_model_1.coef_)
line_model_1.predict(pd.DataFrame({"age": [45]}))

> The intercept moves from 100.8 to 102.8 and the slope from 0.49 to 0.35, and
> the prediction for the 45-year-old from 122.9 to 118.4 mg/dL. Same patients in
> the file, same model, same code: only the choice of training patients changed.

Twenty different splits, twenty different lines:

In [ ]:
fig, ax = plt.subplots()
ax.scatter(glu["age"], glu["glucose"], color="grey")
for s in range(1, 21):
    train_s, _ = train_test_split(glu, test_size=0.2, random_state=s)
    model_s = LinearRegression().fit(train_s[["age"]], train_s["glucose"])
    line_s = ax.plot(ages_line, model_s.intercept_ + model_s.coef_[0] * ages_line,
                     color="steelblue", alpha=0.5)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Glucose (mg/dL)")
ax.set_title("The fitted line from 20 different training splits")
plt.show()

> **Think about it**
>
> Where do the 20 lines agree most, and where do they spread out? What does that
> mean for a prediction at age 25 compared with age 75?

> **Worked answer**
>
> The lines cross near the middle of the data, around the average age and
> average glucose, and fan out towards the ends. Most patients are in their
> twenties and thirties, so there the lines are pinned down by many points. Few
> patients are over 60, so whether one or two of them are in the training set
> swings the slope. A prediction at 75 depends far more on which patients
> happened to be chosen for training than a prediction at 25.

### Part 2 — Add BMI

The same kind of model, with a second predictor:

In [ ]:
plane_model = LinearRegression()
plane_model.fit(train[["age", "BMI"]], train["glucose"])

print("intercept:", plane_model.intercept_)
print("weights (age, BMI):", plane_model.coef_)

$$\widehat{\text{glucose}} = \hat\beta_0 + \hat\beta_1 \times \text{age} + \hat\beta_2 \times \text{BMI}$$

With two predictors the model is no longer a line but a flat **plane** above
the age–BMI floor.

#### Task 2a — The same patient, now with a BMI of 30

> **Stuck?**
>
> Three parameters: the intercept, then the two weights in the order of the
> columns, age first and BMI second.

> **Worked answer**

In [ ]:
predicted_patient = 82.06 + 0.459 * 45 + 0.611 * 30
predicted_patient

Check it with `predict()`:

In [ ]:
patient = pd.DataFrame({"age": [45], "BMI": [30]})

plane_model.predict(patient)

Turn the plot with the mouse and scroll to zoom. The red point is the patient,
sitting on the plane.

In [ ]:
# This plot needs the plotly package: pip install plotly
import plotly.graph_objects as go

# every combination of age and BMI on a fine grid
ages = np.arange(20, 82.5, 0.5)
bmis = np.arange(19, 47.5, 0.5)
A, B = np.meshgrid(ages, bmis)
grid = pd.DataFrame({"age": A.ravel(), "BMI": B.ravel()})

# the plane's prediction at each grid point: one row per BMI, one column per age
plane_matrix = plane_model.predict(grid).reshape(A.shape)

fig_plane = go.Figure()
fig_plane.add_trace(go.Scatter3d(
    x=train["age"], y=train["BMI"], z=train["glucose"], mode="markers",
    marker={"size": 3, "color": "steelblue"}, name="Training patients"))
fig_plane.add_trace(go.Surface(
    x=ages, y=bmis, z=plane_matrix, opacity=0.6, showscale=False,
    colorscale=[[0, "lightgrey"], [1, "grey"]], name="Plane"))
fig_plane.add_trace(go.Scatter3d(
    x=[45], y=[30], z=plane_model.predict(patient), mode="markers",
    marker={"size": 7, "color": "red"}, name="Age 45, BMI 30"))
fig_plane.update_layout(scene={"xaxis_title": "Age (years)",
                               "yaxis_title": "BMI (kg/m²)",
                               "zaxis_title": "Glucose (mg/dL)"})
fig_plane.show()

> **Think about it**
>
> 1. In words, what does the BMI weight mean?
> 2. Adding BMI moved this patient's prediction by how much? Does a second
>    predictor guarantee better predictions?

> **Worked answer**
>
> 1. For two patients of the same age, one BMI unit higher means a predicted
>    glucose about 0.6 mg/dL higher. Each weight is read with the other
>    predictor held fixed.
> 2. From 122.9 to 121.0 mg/dL, about 2 mg/dL. A second predictor gives the
>    model more to work with, but no guarantee: if it carries little
>    information about the outcome, the plane is barely different from the
>    line, and on new patients it can even do slightly worse.

### Part 3 — The same steps, a decision tree

Only the specification changes. The training patients, the predictors and the
`.fit()` and `.predict()` calls stay the same.

In [ ]:
# 1. Specify: a regression tree, kept small enough to follow by hand
tree_model = DecisionTreeRegressor(max_depth=3, min_samples_leaf=7,
                                   random_state=0)

# 2. Fit on the same training patients, with the same predictors
tree_model.fit(train[["age", "BMI"]], train["glucose"])

fig, ax = plt.subplots(figsize=(10, 5))
plot_tree(tree_model, feature_names=["age", "BMI"], filled=True,
          impurity=False, precision=1, ax=ax)
plt.show()

Each box asks a yes/no question. **True goes left, False goes right.** The
`value` in each leaf at the bottom is the prediction: the mean glucose of the
training patients who ended up there. `max_depth=3` allows at most three
questions in a row, and `min_samples_leaf=7` requires at least seven training
patients in every leaf; without them the tree would keep splitting until it was
too large to follow.

#### Task 3a — Follow the patient

Start at the top with the patient aged 45 with a BMI of 30. At each question,
go left or right. Which leaf do you reach, and what glucose does it predict?

> **Worked answer**
>
> `age <= 23.5`? False, go right. `BMI <= 41.0`? True, go left.
> `age <= 26.5`? False, go right. The leaf predicts **130.8** mg/dL: the mean
> glucose of the 29 training patients older than 26.5 with a BMI of at most 41.
>
> The plot rounds the split points to one decimal (`precision=1`); the BMI split
> is exactly 41.05. `print(export_text(tree_model, feature_names=["age", "BMI"]))`
> prints the same tree as text.

Check it with `predict()`:

In [ ]:
tree_model.predict(patient)

Turn the plot. The orange surface is the tree, the grey one the plane from
Part 2.

In [ ]:
# the tree's prediction on the same grid as the plane
tree_matrix = tree_model.predict(grid).reshape(A.shape)

fig_tree = go.Figure()
fig_tree.add_trace(go.Scatter3d(
    x=train["age"], y=train["BMI"], z=train["glucose"], mode="markers",
    marker={"size": 3, "color": "steelblue"}, name="Training patients"))
fig_tree.add_trace(go.Surface(
    x=ages, y=bmis, z=tree_matrix, opacity=0.8, showscale=False,
    colorscale=[[0, "moccasin"], [1, "darkorange"]], name="Tree"))
fig_tree.add_trace(go.Surface(
    x=ages, y=bmis, z=plane_matrix, opacity=0.5, showscale=False,
    colorscale=[[0, "lightgrey"], [1, "grey"]], name="Plane"))
fig_tree.add_trace(go.Scatter3d(
    x=[45], y=[30], z=tree_model.predict(patient), mode="markers",
    marker={"size": 7, "color": "red"}, name="Age 45, BMI 30 (tree)"))
fig_tree.update_layout(scene={"xaxis_title": "Age (years)",
                              "yaxis_title": "BMI (kg/m²)",
                              "zaxis_title": "Glucose (mg/dL)"})
fig_tree.show()

> **Think about it**
>
> 1. Describe the shape of the tree's surface. Why is it made of flat steps?
> 2. Which predictor does the tree ask about first, and how often does it ask
>    about BMI? Compare with how much BMI changed the plane in Part 2.
> 3. A new patient is 90 years old, older than anyone in the training data. What
>    do the plane and the tree predict, and why do they behave differently?

> **Worked answer**
>
> 1. Flat terraces with vertical walls between them. Every patient in the same
>    leaf gets the same prediction, so the surface is constant inside each
>    rectangle of age and BMI and jumps at each split. The plane, by contrast,
>    changes smoothly everywhere.
> 2. Age first, at 23.5, then BMI twice (27.7 for the youngest patients and
>    41.05 for the rest). The tree did use BMI, but the biggest step is between
>    the under-24s and everyone else. The R tab's tree, fitted on other training
>    patients, never asks about BMI at all: whether a weak predictor is used can
>    depend on which patients the tree happened to see.
> 3. The plane keeps rising with age, because a plane has no edge. The tree puts
>    the patient in the oldest leaf and predicts the mean of those training
>    patients, so it never goes above or below what it saw in training:

In [ ]:
old = pd.DataFrame({"age": [90], "BMI": [30]})
print(plane_model.predict(old), tree_model.predict(old))

> **Learning points**
>
> - The steps are the same whatever the model: specify, fit on the training
>   patients, predict for new ones. Swapping linear regression for a tree
>   changed one line.
> - A linear model's prediction can be worked out by hand from its parameters; a
>   tree's prediction is read by following the questions to a leaf.
> - The fitted model depends on which patients were used for training.
> - A line or plane changes smoothly and keeps going beyond the data; a tree
>   predicts in flat steps and stays within the values it saw.